# 18 — Benchmark 2025 v2: campeões do protocolo L=2304 × ano intocado (só inferência)
Avaliação final do protocolo v2: checkpoints de 10–17 (treino 2024, `L=2304 → H=288`) previstos no ano de 2025, nunca tocado. Espelho do 08 (mesmas tabelas/figuras/nomes de modelo), com as diferenças do v2 documentadas abaixo. **Zero treino, zero tuning, zero early-stopping** — só inferência.

## Diff exato vs 08 (`notebooks/08-benchmark-2025.ipynb`)

| | 08 (v1) | 18 (v2, este notebook) |
|---|---|---|
| Janela | `L=8640 → H=288` (30 d) | **`L=2304 → H=288` (8 d, protocolo v2)** |
| Cauda dos modelos | `LN=2016` da janela de 30 d | **cauda `LN=2016` da janela de 8 d (arquiteturas idênticas)** |
| LSTNet | 1 seed, `in_ch=3` (02/03) | **seed-mean 5 seeds, `in_ch=8` (12/13: ToD+solar+Fourier)** |
| PatchTST/DLinear | 1 seed (04/05) | **seed-mean 5 seeds (14/15, univariados)** |
| LGBM | 288 `LGBMRegressor` pickle no resíduo (06/07) | **288 nativos `.txt` no resíduo (16/17; loader tolerante aos dois nomes)** |
| DLinear-res | 1 seed (06/07) | **seed-mean 5 seeds (16/17, resíduo + piso sazonal)** |
| Ensemble | pesos do `ensemble.json` (06/07) | **pesos do `ensemble.json` (16/17, NNLS fit 1–4/report dez)** |
| Prophet | `.json` do 00/01, opcional | **`.json` do 10/11, opcional (mesmo comportamento)** |
| ARIMA | **ausente no 08** (verificado: zero ocorrências de `arima` no notebook) | **omitido como no 08** (ver §5; `arima212_cauda_treino.pkl` do 10/11 é artefato de cauda de treino, sem refit por origem) |
| Lag-365 | copia mesma data de 2024 + fallback saz-288 | **idêntico** |

## Checkpoints exigidos (erro claro se ausente; só Prophet é opcional)

| Variável | LSTNet seed-mean | PatchTST/DLinear seed-mean | LGBM nativo 288 | DLinear-res seed-mean | Ensemble | Prophet (opc.) |
|---|---|---|---|---|---|
| pH | `12-v2-lstnet-ph/modelos/lstnet_ph_s{seed}.pt` ×5 | `14-v2-patchtst-ph/modelos/{patchtst,dlinear}_ph_s{seed}.pt` ×5 | `16-v2-ensemble-ph/modelos/lgbm_nativo/lgbm_h*.txt` | `16-…/dlinear_res_ph_s{seed}.pt` ×5 | `16-…/ensemble.json` | `10-v2-baseline-ph/modelos/prophet_ph.json` |
| OD | `13-v2-lstnet-od/modelos/lstnet_od_s{seed}.pt` ×5 | `15-v2-patchtst-od/modelos/{patchtst,dlinear}_od_s{seed}.pt` ×5 | `17-v2-ensemble-od/modelos/lgbm_nativo/model_j*.txt` | `17-…/dlinear_res_od_s{seed}.pt` ×5 | `17-…/ensemble.json` | `11-v2-baseline-od/modelos/prophet_od.json` |

`seeds = [42, 7, 123, 2024, 999]`. O loader LGBM aceita **ambos** os nomes (`lgbm_h*.txt` do 16 e `model_j*.txt` do 17) e escolhe a convenção das features Fourier pela lista de nomes (`orig_f1sin…` = Fourier do fim do alvo; `f1_sin_orig…` = Fourier da origem `fim−H`).

## Execução remota (UM job por vez — 1 job, sem concorrência)

- Cap de threads quando a máquina estiver compartilhada (`OMP/MKL/OpenBLAS_NUM_THREADS=4`); sem cap quando solo. A célula 0 imprime o estado das variáveis.
- Gargalo: **seed-means torch (4 arquiteturas × 5 seeds × N janelas)** — LSTNet com loop de skip é o mais lento. Medido localmente (CPU): **LSTNet-ph 5 seeds ≈ 7 min p/ 74k janelas; LGBM 288×74k ≈ 1 min/var**. Total estimado solo em CPU: **~25–45 min** (ph 74k + od ~102k janelas, + Prophet + lag-365); compartilhada ≈ 2×. ARIMA por origem está **fora** (como no 08 — um refit por origem dominaria tudo, horas a dias).
- Saídas em `resultados/18-v2-benchmark-2025/`: `metricas_benchmark_{ph,od}.csv` (rolante, primária), `metricas_diaria_{ph,od}.csv`, `metricas_por_dia_{ph,od}.csv`, `metricas_por_mes_{ph,od}.csv` + figs `01-eda`, `04-mae`, `05-diaria`, `06-mensal`, `07-exemplos` (espelho do 08).
- **README só na execução**, com o veredito v1×v2 (réguas v1 do 08: ens pH **0.0509** · ens OD **0.2107**; régua v1/08 dos dias-âncora: pH 0.0503 · OD 0.2072).

In [1]:
import json
import os
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from numpy.lib.stride_tricks import sliding_window_view

import torch
import torch.nn as nn
import lightgbm as lgb

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "univariavel" / "dados" / "treino").exists())
TR = ROOT / "univariavel" / "dados" / "treino"
BM = ROOT / "univariavel" / "dados" / "benchmark"
OUT = ROOT / "univariavel" / "resultados" / "18-v2-benchmark-2025"
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 10–17). Só inferência em 2025.
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
LN, HN = 2016, 288  # cauda nativa dos modelos (7 d dos 8 d)
SEEDS = [42, 7, 123, 2024, 999]
ARIMA_OPCIONAL = False  # 08 não avalia ARIMA no benchmark → espelho = omitir (ver §5)
FILES = {
    "ph": ("ef01-mogi-das-cruzes_ph_2024.csv", "ef01-mogi-das-cruzes_ph_2025.csv", "pH"),
    "od": ("ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv", "ef01-mogi-das-cruzes_oxigenio-dissolvido_2025.csv", "Oxigênio Dissolvido (mg/L)"),
}
CKPT = {
    "ph": {"lstnet_dir": "12-v2-lstnet-ph/modelos", "lstnet_pat": "lstnet_ph_s{seed}.pt",
           "td_dir": "14-v2-patchtst-ph/modelos", "patch_pat": "patchtst_ph_s{seed}.pt",
           "dlin_pat": "dlinear_ph_s{seed}.pt", "lgbm_dir": "16-v2-ensemble-ph/modelos/lgbm_nativo",
           "dlres_pat": "dlinear_res_ph_s{seed}.pt", "ens": "16-v2-ensemble-ph/modelos/ensemble.json",
           "prophet": "10-v2-baseline-ph/modelos/prophet_ph.json"},
    "od": {"lstnet_dir": "13-v2-lstnet-od/modelos", "lstnet_pat": "lstnet_od_s{seed}.pt",
           "td_dir": "15-v2-patchtst-od/modelos", "patch_pat": "patchtst_od_s{seed}.pt",
           "dlin_pat": "dlinear_od_s{seed}.pt", "lgbm_dir": "17-v2-ensemble-od/modelos/lgbm_nativo",
           "dlres_pat": "dlinear_res_od_s{seed}.pt", "ens": "17-v2-ensemble-od/modelos/ensemble.json",
           "prophet": "11-v2-baseline-od/modelos/prophet_od.json"},
}

def exige(p, descr):
    assert p.exists(), (f"checkpoint ausente: {p} ({descr}) — rode o experimento correspondente "
                       "antes ou baixe via bash scripts/baixar_modelos.sh")
    return p

for var, d in CKPT.items():
    exige(ROOT / "univariavel" / "resultados" / d["lstnet_dir"] / d["lstnet_pat"].format(seed=SEEDS[0]), f"{var}/lstnet s{SEEDS[0]}")
    for sd in SEEDS:
        exige(ROOT / "univariavel" / "resultados" / d["lstnet_dir"] / d["lstnet_pat"].format(seed=sd), f"{var}/lstnet s{sd}")
        exige(ROOT / "univariavel" / "resultados" / d["td_dir"] / d["patch_pat"].format(seed=sd), f"{var}/patchtst s{sd}")
        exige(ROOT / "univariavel" / "resultados" / d["td_dir"] / d["dlin_pat"].format(seed=sd), f"{var}/dlinear s{sd}")
        exige(ROOT / "univariavel" / "resultados" / Path(d["lgbm_dir"]).parent / d["dlres_pat"].format(seed=sd), f"{var}/dlres s{sd}")
    nh = len(list((ROOT / "univariavel" / "resultados" / d["lgbm_dir"]).glob("lgbm_h*.txt")))
    nj = len(list((ROOT / "univariavel" / "resultados" / d["lgbm_dir"]).glob("model_j*.txt")))
    assert (nh == H) ^ (nj == H), (f"{var}/lgbm: esperado 288 boosters de UM padrão, "
                                     f"achei lgbm_h*={nh} model_j*={nj} em {d['lgbm_dir']}")
    exige(ROOT / "univariavel" / "resultados" / d["ens"], f"{var}/ensemble.json")
    pp = ROOT / "univariavel" / "resultados" / d["prophet"]
    print(f"{var}: torch×4×5seeds + lgbm({nh or nj} boosters) + ensemble.json OK | "
          f"prophet: {'OK' if pp.exists() else 'AUSENTE (opcional, será pulado)'}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count(), "| torch:", torch.__version__)
print("estimativa solo CPU: ~25–45 min (4 torch×5 seeds dominam; LGBM ~1 min/var; sem ARIMA, como no 08)")
print("UM job por vez nesta máquina — sem concorrência com outros treinos")

ph: torch×4×5seeds + lgbm(288 boosters) + ensemble.json OK | prophet: OK
od: torch×4×5seeds + lgbm(288 boosters) + ensemble.json OK | prophet: OK
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12 | torch: 2.14.0+cpu
estimativa solo CPU: ~25–45 min (4 torch×5 seeds dominam; LGBM ~1 min/var; sem ARIMA, como no 08)
UM job por vez nesta máquina — sem concorrência com outros treinos


## 1. Carga 2024 (fonte do lag-365) + 2025 (benchmark), limpeza idêntica

In [2]:
def ler(path, col):
    df = pd.read_csv(path, sep=";", decimal=",", encoding="windows-1252",
                     skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
    df = df.rename(columns={"Data hora": "ds", col: "y"}).sort_values("ds").reset_index(drop=True)
    idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
    s_raw = df.set_index("ds")["y"].reindex(idx)
    s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
    return s, s_raw

series = {}
for var, (f24, f25, col) in FILES.items():
    s24, _ = ler(TR / f24, col)
    s25, r25 = ler(BM / f25, col)
    series[var] = {"s24": s24, "s": s25, "raw": r25}
    print(f"{var}: treino {s24.shape} NaN={int(s24.isna().sum())} | benchmark {s25.shape} NaN={int(s25.isna().sum())}")

for var, d in series.items():
    s = d["s"]
    fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
    ax[0].plot(s.index, s.values, lw=0.3)
    ax[0].set_title(f"{var.upper()} 2025 — benchmark (ano intocado até aqui)")
    s.hist(bins=60, ax=ax[1])
    ax[1].set_title("Distribuição 2025")
    pd.Series(s.values, index=s.index).groupby(s.index.hour).mean().plot(ax=ax[2])
    ax[2].set_title("Ciclo diário médio 2025")
    ax[2].set_xlabel("hora")
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"01-eda-{var}.png")
print("figs salvas")

ph: treino (105121,) NaN=6408 | benchmark (104833,) NaN=544


od: treino (105121,) NaN=336 | benchmark (104833,) NaN=129


figs salvas


## 2. Janelamento do benchmark (L=2304) + âncoras diárias (ano todo)

In [3]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

bench = {}
for var, d in series.items():
    s = d["s"]
    v = s.to_numpy().astype(np.float32)
    W = sliding_window_view(v, L + H)
    ok = ~np.isnan(W).any(axis=1)
    W = W[ok]
    X, Y = W[:, :L], W[:, L:]
    ends = s.index[L + H - 1:][ok]
    di = np.where(ends.time == pd.Timestamp("23:55").time())[0]
    bench[var] = {"X": X, "Y": Y, "ends": ends, "di": di, "s": s}
    print(f"{var}: {len(X)} janelas | dias-âncora: {len(di)} ({ends[di[0]].date()} → {ends[di[-1]].date()})")

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

ph: 74222 janelas | dias-âncora: 259 (2025-01-09 → 2025-12-30)


od: 86567 janelas | dias-âncora: 302 (2025-01-09 → 2025-12-30)


## 3. Sazonal lag-365 (copia a mesma data de 2024; fallback p/ saz-288 onde 2024 falha)

In [4]:
lag365 = {}
for var, d in series.items():
    s24, X, Y, ends = d["s24"], bench[var]["X"], bench[var]["Y"], bench[var]["ends"]
    P = np.empty_like(Y)
    fb = 0
    s24v = s24.to_numpy()
    s24i = s24.index
    for k in range(len(Y)):
        e = ends[k]
        try:
            src_end = s24i.get_loc(pd.Timestamp(year=2024, month=e.month, day=e.day,
                                                hour=e.hour, minute=e.minute))
            src = s24v[src_end-287:src_end+1]
        except KeyError:
            src = None
        if src is None or np.isnan(src).any():
            src = X[k][L-SEASON:L]  # fallback honesto: saz-288
            fb += 1
        P[k] = src
    lag365[var] = P
    print(f"{var}: lag-365 pronto | fallback saz-288 em {fb}/{len(Y)} janelas ({100*fb/len(Y):.1f}%)")
    print(f"  lag-365 rolante:", {k: round(v, 4) for k, v in metricas(Y, P).items()})

ph: lag-365 pronto | fallback saz-288 em 3350/74222 janelas (4.5%)


  lag-365 rolante: {'MAE': 0.3692, 'RMSE': 0.4495, 'MAPE': 5.9364, 'sMAPE': 6.1952}


od: lag-365 pronto | fallback saz-288 em 939/86567 janelas (1.1%)


  lag-365 rolante: {'MAE': 1.1353, 'RMSE': 1.4142, 'MAPE': 21.6136, 'sMAPE': 26.2689}


## 4. Arquiteturas v2 (cópias fiéis de 12/13/14/15/16/17) + covariáveis determinísticas

- `LSTNet1D`: verbatim do 12/13 (`Conv1d 8→32/k12/s6 · GRU 64 · skip GRUCell 32/p48 · head 96→288 · AR 288→288`, RevIN por janela). Entrada: cauda `LN=2016` + 7 canais (`tod_sin/cos`, solar/90, `f1..f4` do dia-do-ano — futuro conhecido, sem leakage; funções verbatim do 12/16).
- `PatchTST` / `DLinearFull`: verbatim do 14/15 (univariados, cauda `LN=2016`, RevIN por janela).
- `DLinearRes`: verbatim do 16/17 (prevê o **resíduo** `Y − sazonal`; o piso é somado fora).
- **ARIMA**: o 08 não avalia ARIMA no benchmark (verificado por leitura — zero ocorrências). Espelho = omitir o refit por origem aqui também. `arima_hora_10` abaixo replica o helper do 10 §7 (grade horária 720 h/24 h repeat ×12) para UMA origem, mas só roda com `ARIMA_OPCIONAL=True` (padrão `False`).

In [5]:
# --- covariáveis determinísticas (verbatim do 12/16; Mogi das Cruzes −23,52/−46,19, UTC−3) ---
LAT, LON, TZ = -23.52, -46.19, -3
N_COV, N_CH = 7, 8
CONV_CH, CONV_K, CONV_S = 32, 12, 6
GRU_H, SKIP_H, SKIP_P = 64, 32, 48
AR_Q = 288
PATCH_P, PATCH_S = 48, 24
D_MODEL, NLAYERS, NHEAD, FF = 64, 3, 4, 128
DROPOUT = 0.1


def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))


def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))


class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(N_CH, CONV_CH, kernel_size=CONV_K, stride=CONV_S)
        self.gru = nn.GRU(CONV_CH, GRU_H, batch_first=True)
        self.skipcell = nn.GRUCell(CONV_CH, SKIP_H)
        self.head = nn.Linear(GRU_H + SKIP_H, HN)
        self.ar = nn.Linear(AR_Q, HN)
        self.drop = nn.Dropout(DROPOUT)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, SKIP_H, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - SKIP_P] if t - SKIP_P >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -AR_Q:])
        return (yn + ya - self.beta) / g * sg + mu


class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - PATCH_P) // PATCH_S + 1
        self.proj = nn.Linear(PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, PATCH_P, PATCH_S)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


class DLinearFull(nn.Module):  # 14/15: previsão direta (verbatim do 14 §8)
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


class DLinearRes(nn.Module):  # 16/17: prevê o RESÍDUO Y − sazonal (verbatim do 16 §10)
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg


def arima_hora_10(e, hs):
    """Helper do 10 §7 (grade horária 720 h/24 h repeat ×12) p/ UMA origem. Não usado por padrão."""
    from statsmodels.tsa.arima.model import ARIMA
    he = e.floor("h")
    ctx = hs.loc[he - pd.Timedelta(hours=719):he].values
    fc = ARIMA(ctx, order=(2, 1, 2)).fit().get_forecast(24).predicted_mean.values
    return np.repeat(fc, 12)[:H]

print(f"arquiteturas ok: lstnet={sum(p.numel() for p in LSTNet1D().parameters())} "
      f"patchtst={sum(p.numel() for p in PatchTST().parameters())} "
      f"dlinear={sum(p.numel() for p in DLinearFull().parameters())}")

arquiteturas ok: lstnet=139426 patchtst=1639010 dlinear=1161794


## 5. Inferência por variável (cheap + lag365 + prophet opcional + seed-means + lgbm + dlres + ens)

In [6]:
def base_feats(Xb, E):  # verbatim do 06/07/16/17 (cabe em L=2304: máx lag 2016, fase L−288k ≥ 288)
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):  # verbatim do 06/07: hora (inteira) do passo-alvo j a partir do fim do alvo
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

def solar_passo(E, j):  # elevação solar/90 do passo-alvo j (espelho exato de hour_sincos)
    return (elevacao_solar(E - pd.to_timedelta((H - 1 - j)*5, unit="min")) / 90.0).astype(np.float32)

def build_cov(s):  # 7 canais determinísticos (verbatim do 12/16 §7)
    tod_s = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
    tod_c = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
    sol = (elevacao_solar(s.index) / 90.0).astype(np.float32)
    f1, f2, f3, f4 = [a.astype(np.float32) for a in fourier_doy(s.index)]
    return np.stack([tod_s, tod_c, sol, f1, f2, f3, f4], axis=1).astype(np.float32)

def carrega_lgbm(nat_dir, ens_path):
    """Loader tolerante: aceita lgbm_h*.txt (16) e model_j*.txt (17).
    Convenção Fourier decidida pela lista de nomes (normalizacao/ensemble) ou pelo glob."""
    h = sorted(nat_dir.glob("lgbm_h*.txt"))
    j = sorted(nat_dir.glob("model_j*.txt"))
    assert (len(h) == H) ^ (len(j) == H), (f"esperado 288 boosters de UM padrão em {nat_dir}: "
                                             f"lgbm_h*={len(h)} model_j*={len(j)}")
    paths, glob_flavor = (h, "fim") if len(h) == H else (j, "ctx")
    flavor = glob_flavor
    for key in ("lgbm_features", "features"):
        for src in (nat_dir.parent / "normalizacao.json", ens_path):
            try:
                meta = json.load(open(src))
            except (FileNotFoundError, json.JSONDecodeError):
                continue
            nomes = meta.get(key) or (meta.get("lgbm") or {}).get(key)
            if nomes:
                if "orig_f1sin" in nomes:
                    flavor = "fim"  # 16: fourier do fim do alvo (E)
                elif "f1_sin_orig" in nomes:
                    flavor = "ctx"  # 17: fourier da origem (E − H)
                break
    assert flavor == glob_flavor, f"nomes × arquivos divergem em {nat_dir}: {flavor} vs {glob_flavor}"
    boosters = [lgb.Booster(model_file=str(p)) for p in paths]
    print(f"  lgbm: {len(boosters)} boosters ({paths[0].name}…{paths[-1].name}, fourier={flavor})")
    return boosters, flavor

def prevê_lgbm(boosters, flavor, Xb, E):
    F, em = base_feats(Xb, E)
    if flavor == "fim":
        Ff = np.column_stack([a.astype(np.float32) for a in fourier_doy(E)])
    else:
        Ff = np.column_stack([a.astype(np.float32) for a in fourier_doy(E - pd.Timedelta(minutes=5*H))])
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(Xb), H), dtype=np.float32)
    for j, bst in enumerate(boosters):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] + bst.predict(np.column_stack([F, Ff, sh, ch, solar_passo(E, j)]))
    return P

@torch.no_grad()
def seed_mean(modelos, Wln, Tln, rowln, idxs, batch=256, com_cov=True):
    acc = None
    for m in modelos:
        m.eval()
        outs = []
        for b in range(0, len(idxs), batch):
            r = rowln[idxs[b:b+batch]]
            xb = torch.from_numpy(Wln[r])
            if com_cov:
                outs.append(m(xb, torch.from_numpy(Tln[r])).numpy())
            else:
                outs.append(m(xb).numpy())
        P = np.concatenate(outs)
        acc = P if acc is None else acc + P
    return (acc / len(modelos)).astype(np.float32)

@torch.no_grad()
def seed_mean_res(modelos, Xb, batch=512):
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    Xt = torch.from_numpy(Xb[:, -LN:].astype(np.float32))
    acc = np.zeros((len(Xb), H), dtype=np.float32)
    for m in modelos.values():
        m.eval()
        outs = []
        for b in range(0, len(Xt), batch):
            outs.append(m(Xt[b:b+batch]).numpy())
        acc += S + np.concatenate(outs)
    return (acc / len(modelos)).astype(np.float32)

def load_state(cls, path, **kw):
    m = cls(**kw).to("cpu")
    m.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["state"])
    return m.eval()

resultados = {}
for var in ["ph", "od"]:
    t0 = time.time()
    C = CKPT[var]
    X, Y, ends = bench[var]["X"], bench[var]["Y"], bench[var]["ends"]
    s = bench[var]["s"]
    preds = cheap_preds(X)
    preds["sazonal_lag365"] = lag365[var]
    S = preds["sazonal_naive_288"]
    # --- janelas nativas LN=2016 (construção do 12/14: cauda da janela L=2304) ---
    val5 = s.to_numpy().astype(np.float32)
    COV = build_cov(s)
    Wln = sliding_window_view(val5, LN)
    Tln = sliding_window_view(COV, LN, axis=0).transpose(0, 2, 1).astype(np.float32)
    rowln = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H)) - LN + 1
    assert Wln.shape[1] == LN and Tln.shape[1:] == (LN, N_COV) and (rowln >= 0).all()
    idx = np.arange(len(X))
    # --- LSTNet seed-mean (12/13) ---
    lstms = [load_state(LSTNet1D, ROOT / "univariavel" / "resultados" / C["lstnet_dir"] / C["lstnet_pat"].format(seed=sd))
             for sd in SEEDS]
    Pn = seed_mean(lstms, Wln, Tln, rowln, idx)
    del lstms
    # --- PatchTST/DLinear seed-mean (14/15, univariados) ---
    ptms = [load_state(PatchTST, ROOT / "univariavel" / "resultados" / C["td_dir"] / C["patch_pat"].format(seed=sd))
            for sd in SEEDS]
    Pt = seed_mean(ptms, Wln, None, rowln, idx, com_cov=False)
    del ptms
    dlms = [load_state(DLinearFull, ROOT / "univariavel" / "resultados" / C["td_dir"] / C["dlin_pat"].format(seed=sd))
            for sd in SEEDS]
    Dl = seed_mean(dlms, Wln, None, rowln, idx, com_cov=False)
    del dlms
    # --- LGBM nativo 288 (16/17) + DLinear-res seed-mean (16/17) ---
    boosters, flavor = carrega_lgbm(ROOT / "univariavel" / "resultados" / C["lgbm_dir"], ROOT / "univariavel" / "resultados" / C["ens"])
    Gb = prevê_lgbm(boosters, flavor, X, ends)
    del boosters
    drms = {sd: load_state(DLinearRes, (ROOT / "univariavel" / "resultados" / C["lgbm_dir"]).parent / C["dlres_pat"].format(seed=sd))
            for sd in SEEDS}
    Dr = seed_mean_res(drms, X)
    del drms
    # --- ensemble via ensemble.json (pesos NNLS do 16/17; chaves estáveis) ---
    pesos = json.load(open(ROOT / "univariavel" / "resultados" / C["ens"]))["pesos"]
    En = (pesos.get("sazonal", 0.0) * S + pesos.get("lstnet", 0.0) * Pn
          + pesos.get("lgbm", 0.0) * Gb + pesos.get("dlres", 0.0) * Dr).astype(np.float32)
    print(f"{var}: pesos ensemble { {k: pesos.get(k, 0.0) for k in ['sazonal', 'lstnet', 'lgbm', 'dlres']} }")
    preds.update({"lstnet": Pn, "patchtst": Pt, "dlinear": Dl, "lgbm": Gb, "dlres": Dr, "ens": En})
    # --- Prophet opcional (reaproveita o .json do 10/11; fmap vetorizado = mesmos valores do loop do 08) ---
    Pp = ROOT / "univariavel" / "resultados" / C["prophet"]
    if Pp.exists():
        try:
            from prophet.serialize import model_from_json
            m = model_from_json(Pp.read_text())
            fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]
            F = fmap.reindex(s.index).to_numpy(dtype=np.float32)
            epos = s.index.get_indexer(ends)
            assert (epos >= 0).all() and not np.isnan(F).all()
            preds["prophet"] = F[epos[:, None] - (H - 1 - np.arange(H))[None, :]].astype(np.float32)
            print(f"{var}: prophet incluído")
        except Exception as e:
            print(f"{var}: prophet pulado ({type(e).__name__}: {str(e)[:150]})")
    else:
        print(f"{var}: prophet ausente — pulado (opcional)")
    if ARIMA_OPCIONAL:  # fora por padrão: o 08 não avalia ARIMA no benchmark
        hs = s.resample("1h").mean()
        preds["arima_212_h"] = np.stack([arima_hora_10(e, hs) for e in ends])
    resultados[var] = preds
    print(f"{var}: inferência em {time.time()-t0:.0f}s | modelos: {sorted(preds)}")
    del COV, Wln, Tln, Gb, X, Y

  lgbm: 288 boosters (lgbm_h000.txt…lgbm_h287.txt, fourier=fim)


Importing plotly failed. Interactive plots will not work.


ph: pesos ensemble {'sazonal': 0.0694, 'lstnet': 0.7048, 'lgbm': 0.0889, 'dlres': 0.1366}


ph: prophet incluído
ph: inferência em 350s | modelos: ['dlinear', 'dlres', 'ens', 'lgbm', 'lstnet', 'media_movel_288', 'patchtst', 'persistencia', 'prophet', 'sazonal_lag365', 'sazonal_naive_288']


  lgbm: 288 boosters (model_j000.txt…model_j287.txt, fourier=ctx)


od: pesos ensemble {'sazonal': 0.0038, 'lstnet': 0.6771, 'lgbm': 0.0, 'dlres': 0.3216}


od: prophet incluído
od: inferência em 398s | modelos: ['dlinear', 'dlres', 'ens', 'lgbm', 'lstnet', 'media_movel_288', 'patchtst', 'persistencia', 'prophet', 'sazonal_lag365', 'sazonal_naive_288']


## 6. Tabelas (rolante + dias-âncora + por dia + por mês)

In [7]:
for var in ["ph", "od"]:
    X, Y, ends, di = bench[var]["X"], bench[var]["Y"], bench[var]["ends"], bench[var]["di"]
    tab = pd.DataFrame({m: metricas(Y, p) for m, p in resultados[var].items()}).T.round(4)
    tab.to_csv(OUT / f"metricas_benchmark_{var}.csv")
    print(f"=== {var} benchmark rolante ({len(X)} origens) ===")
    print(tab.to_string())
    Yd = Y[di]
    tab_d = pd.DataFrame({m: metricas(Yd, p[di]) for m, p in resultados[var].items()}).T.round(4)
    tab_d.to_csv(OUT / f"metricas_diaria_{var}.csv")
    print(f"=== {var} dias-âncora ({len(di)}) ===")
    print(tab_d.to_string())
    por_dia = pd.DataFrame({m: [mae(Yd[k:k+1], p[di][k:k+1]) for k in range(len(di))]
                            for m, p in resultados[var].items()},
                           index=[str(ends[i].date()) for i in di])
    por_dia.to_csv(OUT / f"metricas_por_dia_{var}.csv")
    meses = pd.to_datetime(por_dia.index).month
    por_mes = por_dia.groupby(meses).mean().round(4)
    nomes_mes = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
    por_mes.index = [nomes_mes[m-1] for m in por_mes.index]  # só meses com âncora (outages removem meses)
    por_mes.to_csv(OUT / f"metricas_por_mes_{var}.csv")
    print(f"=== {var} MAE médio por mês ===")
    print(por_mes.to_string())
    print(f"\nRégua benchmark v2 {var}: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}\n")

=== ph benchmark rolante (74222 origens) ===
                      MAE    RMSE     MAPE    sMAPE
persistencia       0.0746  0.1009   1.2063   1.2056
sazonal_naive_288  0.0565  0.0788   0.9167   0.9158
media_movel_288    0.0632  0.0829   1.0233   1.0222
sazonal_lag365     0.3692  0.4495   5.9364   6.1952
lstnet             0.0476  0.0655   0.7714   0.7712
patchtst           0.0465  0.0644   0.7539   0.7536
dlinear            0.0472  0.0657   0.7660   0.7655
lgbm               0.0768  0.1004   1.2405   1.2475
dlres              0.0486  0.0684   0.7886   0.7881
ens                0.0470  0.0646   0.7620   0.7622
prophet            1.1659  1.3271  18.7135  21.3237
=== ph dias-âncora (259) ===
                      MAE    RMSE     MAPE    sMAPE
persistencia       0.0793  0.1048   1.2864   1.2770
sazonal_naive_288  0.0566  0.0788   0.9185   0.9177
media_movel_288    0.0623  0.0820   1.0099   1.0090
sazonal_lag365     0.3684  0.4485   5.9231   6.1799
lstnet             0.0486  0.0661   0.7883

=== ph MAE médio por mês ===
     persistencia  sazonal_naive_288  media_movel_288  sazonal_lag365  lstnet  patchtst  dlinear    lgbm   dlres     ens  prophet
jan        0.0775             0.0754           0.0636          0.1640  0.0596    0.0569   0.0576  0.0728  0.0569  0.0584   0.1141
fev        0.0666             0.0468           0.0534          0.4280  0.0420    0.0378   0.0398  0.1183  0.0384  0.0400   0.4611
mar        0.0755             0.0447           0.0631          0.1822  0.0416    0.0367   0.0359  0.0551  0.0351  0.0372   0.5510
abr        0.0586             0.0611           0.0635          0.2630  0.0506    0.0472   0.0463  0.0580  0.0475  0.0499   0.6487
mai        0.0464             0.0420           0.0397          0.5275  0.0367    0.0386   0.0406  0.0413  0.0387  0.0356   0.7693
jun        0.0357             0.0264           0.0220          0.4882  0.0212    0.0204   0.0205  0.0629  0.0222  0.0202   1.4357
jul        0.0558             0.0289           0.0359        

=== od benchmark rolante (86567 origens) ===
                      MAE    RMSE     MAPE     sMAPE
persistencia       0.5181  0.7005   9.6096    9.4822
sazonal_naive_288  0.2519  0.3739   4.9670    4.8858
media_movel_288    0.4501  0.5591   8.4914    8.3566
sazonal_lag365     1.1353  1.4142  21.6136   26.2689
lstnet             0.2330  0.3286   4.6476    4.5724
patchtst           0.2056  0.3031   4.1173    4.0538
dlinear            0.2073  0.3102   4.1051    4.0629
lgbm               0.3070  0.4137   5.8191    5.8795
dlres              0.2074  0.3111   4.1066    4.0642
ens                0.2133  0.3100   4.2521    4.1754
prophet            5.3414  6.1357  88.8832  138.6936
=== od dias-âncora (302) ===
                      MAE    RMSE     MAPE     sMAPE
persistencia       0.6116  0.7874  11.8578   10.8786
sazonal_naive_288  0.2527  0.3754   4.9952    4.9111
media_movel_288    0.4435  0.5533   8.3733    8.2505
sazonal_lag365     1.1353  1.4143  21.6350   26.2938
lstnet             0.2337

=== od MAE médio por mês ===
     persistencia  sazonal_naive_288  media_movel_288  sazonal_lag365  lstnet  patchtst  dlinear    lgbm   dlres     ens  prophet
jan        0.5690             0.2916           0.4330          1.5574  0.2817    0.2500   0.2260  0.2864  0.2218  0.2377   0.7718
fev        0.5406             0.1785           0.4133          1.6441  0.2263    0.1847   0.1740  0.2265  0.1657  0.1912   0.8184
mar        0.8376             0.2086           0.6384          2.3737  0.2263    0.1956   0.1912  0.2587  0.1788  0.1836   1.6894
abr        0.4850             0.3497           0.4333          1.0422  0.3328    0.3237   0.3290  0.3308  0.3281  0.3050   1.6264
mai        0.3868             0.3706           0.3597          1.4375  0.3202    0.3202   0.3320  0.3601  0.3393  0.3147   3.0261
jun        0.2379             0.1576           0.1783          1.2870  0.1462    0.1373   0.1190  0.2575  0.1201  0.1353   5.2652
jul        0.4162             0.1492           0.2718        

## 7. Figuras (espelho do 08)

In [8]:
for var in ["ph", "od"]:
    tab = pd.read_csv(OUT / f"metricas_benchmark_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    tab["MAE"].sort_values().plot.barh(ax=ax)
    ax.set_title(f"MAE benchmark 2025 v2 — {var} (menor = melhor)")
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"04-mae-{var}.png")

    pord = pd.read_csv(OUT / f"metricas_por_dia_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(12, 3.5))
    for col in ["sazonal_lag365", "sazonal_naive_288", "lstnet", "ens"]:
        if col in pord.columns:
            ax.plot(pd.to_datetime(pord.index), pord[col], lw=1.1, label=col)
    ax.set_title(f"{var} — MAE por dia em 2025 (v2)")
    ax.legend(fontsize=8); fig.autofmt_xdate()
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"05-diaria-{var}.png")

    porm = pd.read_csv(OUT / f"metricas_por_mes_{var}.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    for col in ["sazonal_lag365", "sazonal_naive_288", "lstnet", "ens"]:
        if col in porm.columns:
            ax.plot(porm.index, porm[col], marker="o", ms=3, lw=1.2, label=col)
    ax.set_title(f"{var} — MAE médio por mês em 2025 v2 (sazonalidade do erro)")
    ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"06-mensal-{var}.png")

    X, Y, ends, di = bench[var]["X"], bench[var]["Y"], bench[var]["ends"], bench[var]["di"]
    assert len(di) >= 3, f"dias-âncora insuficientes em {var}: {len(di)}"
    ks = [0, len(di)//2, -1]
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    for ax, k in zip(axes, ks):
        tf = pd.date_range(ends[di[k]] - pd.Timedelta(minutes=5*(H-1)), ends[di[k]], freq="5min")
        ax.plot(tf, Y[di[k]], "k-", lw=1.2, label="real")
        ax.plot(tf, resultados[var]["sazonal_lag365"][di[k]], "--", lw=1, label="lag-365")
        ax.plot(tf, resultados[var]["sazonal_naive_288"][di[k]], ":", lw=1, label="saz-288")
        ax.plot(tf, resultados[var]["ens"][di[k]], lw=1, alpha=0.9, label="ens")
        ax.set_title(f"{var} dia {ends[di[k]].date()}")
        ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / f"07-exemplos-{var}.png")
print("figs salvas")

figs salvas


## 8. Veredito
Réguas v2 do benchmark 2025 acima (célula 6): rolante pH patchtst **0.0465** (ens 0.0470) · OD patchtst **0.2056** (ens 0.2133); dias-âncora pH ens 0.0463 · OD dlres 0.2072. Este notebook não treina nada. Procedência: temporal-remote 192.168.1.6, dir /home/marcos/temporal-model, 2026-09-17 (inferência ph 350 s + od 398 s, 12c). Ver resultados/18-v2-benchmark-2025/README.md (veredito v1×v2).